# YouTube Comment Toxicity Analysis
**Query:** `Wahlbetrug Deutschland` (Election Fraud Germany)  
**Model:** `EIStakovskii/german_toxicity_classifier_plus_v2`  
**Platform:** YouTube Data API v3

---
This notebook collects YouTube comments using a politically sensitive search query and classifies them using a German-language BERT-based toxicity model.

## 1. Setup & Authentication

In [ ]:
from googleapiclient.discovery import build
import pandas as pd

# Set your YouTube Data API v3 key here
# Obtain one at: https://console.cloud.google.com/
API_KEY = "YOUR_YOUTUBE_API_KEY"

youtube = build('youtube', 'v3', developerKey=API_KEY)

## 2. Video Search
Search for German-language videos related to election fraud claims.

In [ ]:
# Search for videos matching the query, filtered to Germany
request = youtube.search().list(
    q="Wahlbetrug Deutschland",
    part="snippet",
    maxResults=10,
    regionCode="DE",
    relevanceLanguage="de"
)
results = request.execute()

# Extract video IDs from search results
video_ids = [
    item['id']['videoId']
    for item in results['items']
    if item['id']['kind'] == 'youtube#video'
]

print(f"Videos found: {len(video_ids)}")
print(video_ids)

## 3. Comment Collection
Fetch top-level comments for each video. Videos with disabled comments are skipped.

In [ ]:
comments = []

for video_id in video_ids:
    try:
        response = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,
            textFormat="plainText"
        ).execute()

        for item in response['items']:
            comment = item['snippet']['topLevelComment']['snippet']
            comments.append({
                'video_id': video_id,
                'text': comment['textDisplay'],
                'likes': comment['likeCount'],
                'date': comment['publishedAt']
            })

    except Exception:
        # Skip videos with comments disabled (403 commentsDisabled)
        pass

df = pd.DataFrame(comments)
print(f"Total comments collected: {len(df)}")
df.head()

## 4. Toxicity Classification
Apply a German BERT-based toxicity classifier to each comment.  
Model: [`EIStakovskii/german_toxicity_classifier_plus_v2`](https://huggingface.co/EIStakovskii/german_toxicity_classifier_plus_v2)  
Labels: `toxic` / `neutral`

In [ ]:
from transformers import pipeline

# Load the German toxicity classification model
classifier = pipeline(
    "text-classification",
    model="EIStakovskii/german_toxicity_classifier_plus_v2"
)

# Classify each comment (truncate to 512 tokens for BERT input limit)
df['toxicity'] = df['text'].apply(
    lambda x: classifier(x[:512])[0]['label']
)

# Summary of toxicity distribution
print("Toxicity label distribution:")
df['toxicity'].value_counts()

## 5. Visualization
Plot the overall toxicity distribution and temporal trend of toxic comments.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Left: Toxicity distribution bar chart ---
counts = df['toxicity'].value_counts()
axes[0].bar(
    counts.index,
    counts.values,
    color=['#e74c3c', '#2ecc71'],
    edgecolor='white',
    linewidth=1.5
)
axes[0].set_title("Toxicity Distribution\n(Wahlbetrug Deutschland)", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Comment Count", fontsize=11)
axes[0].set_xlabel("Label", fontsize=11)
for i, (label, val) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, val + 1, str(val), ha='center', fontsize=11, fontweight='bold')

# --- Right: Temporal trend of toxic comments ---
df['date'] = pd.to_datetime(df['date'], utc=True)
df['month'] = df['date'].dt.to_period('M')

toxic_trend = df[df['toxicity'] == 'toxic'].groupby('month').size()
toxic_trend.plot(
    kind='line',
    ax=axes[1],
    color='#e74c3c',
    marker='o',
    linewidth=2
)
axes[1].set_title("Toxic Comments Over Time", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Month", fontsize=11)
axes[1].set_ylabel("Toxic Comment Count", fontsize=11)
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("toxicity_analysis.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as toxicity_analysis.png")

## 6. Export Results

In [ ]:
# Save the classified comments to CSV
output_path = "youtube_comments.csv"
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"Saved {len(df)} comments to {output_path}")

# Quick summary stats
total = len(df)
toxic_count = (df['toxicity'] == 'toxic').sum()
print(f"\n--- Summary ---")
print(f"Total comments : {total}")
print(f"Toxic          : {toxic_count} ({toxic_count/total*100:.1f}%)")
print(f"Neutral        : {total - toxic_count} ({(total - toxic_count)/total*100:.1f}%)")